# 02 — ETL Pipeline Walkthrough

Runs each stage of `app/etl/` on the raw data and inspects what changed. The point is to
verify the pipeline does what it claims — every rule here is also asserted in
`tests/test_etl.py`, so this notebook is the visual companion to that suite.

In [ ]:
# Run from the repo root or from ai-service/ — this resolves either way.
import sys, pathlib

root = pathlib.Path.cwd()
while root != root.parent and not (root / 'app' / 'etl').exists():
    root = root.parent
sys.path.insert(0, str(root))
print('ai-service root:', root)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 40)
sns.set_theme(style='whitegrid')

In [ ]:
from app.etl.extract import extract
from app.etl.clean import (
    clean, remove_duplicates, handle_missing_values,
    cap_outliers, validate_types, check_consistency,
)
from app.etl.transform import (
    engineer_features, scale_features,
    BASE_FEATURES, DERIVED_FEATURES, ALL_FEATURES,
)
from app.etl.validate import validate

import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

## Stage 1 — Extract

In [ ]:
raw = pd.read_csv(root / 'data' / 'raw' / 'hr_training_data.csv')
print(f'{len(raw):,} rows x {raw.shape[1]} columns')
raw.head(3)

## Stage 2 — Clean

Five rules, applied in order. Each is run separately here so the row-level effect is visible.

In [ ]:
step = raw.copy()
trace = []

for name, fn in [
    ('remove_duplicates', remove_duplicates),
    ('handle_missing_values', handle_missing_values),
    ('cap_outliers', cap_outliers),
    ('validate_types', validate_types),
    ('check_consistency', check_consistency),
]:
    before_rows, before_nulls = len(step), int(step.isnull().sum().sum())
    step = fn(step)
    trace.append({
        'stage': name,
        'rows_before': before_rows, 'rows_after': len(step),
        'nulls_before': before_nulls, 'nulls_after': int(step.isnull().sum().sum()),
    })

pd.DataFrame(trace)

### Did the cleaning rules hold?

The same assertions as `TestCleanPipeline::test_output_satisfies_every_rule_at_once`.

In [ ]:
cleaned = clean(raw)

checks = {
    'no nulls remain': not cleaned.isnull().any().any(),
    'engagement in [1, 5]': cleaned['engagementScore'].between(1, 5).all(),
    'performance in [1, 5]': cleaned['performanceScore'].between(1, 5).all(),
    'absenteeism >= 0': cleaned['absenteeismDays'].min() >= 0,
    'tenure >= 1': cleaned['tenureMonths'].min() >= 1,
    'promotion <= tenure': (cleaned['lastPromotionMonths'] <= cleaned['tenureMonths']).all(),
}
for rule, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {rule}")
assert all(checks.values())

### Capping in effect

Before and after for the features with the heaviest tails.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, col in zip(axes.ravel(), ['salary', 'overtimeHours', 'absenteeismDays', 'trainingHours']):
    sns.kdeplot(raw[col].dropna(), ax=ax, label='raw', fill=True, alpha=0.3, color='#ff4d4f')
    sns.kdeplot(cleaned[col], ax=ax, label='cleaned', fill=True, alpha=0.3, color='#52c41a')
    ax.set_title(col); ax.set_xlabel(''); ax.legend()
plt.tight_layout()
plt.show()

## Stage 3 — Transform

### Feature engineering

Four ratio features. Each divisor is clipped, because a zero denominator would produce `inf`
and poison the scaler:

| Feature | Formula | Reads as |
|---|---|---|
| `salary_per_tenure` | `salary / max(tenure, 1)` | pay relative to time served |
| `engagement_performance` | `engagement / max(performance, 0.1)` | engaged but underperforming, or the reverse |
| `overtime_absenteeism` | `overtime / max(absent + 1, 1)` | working long hours *and* present |
| `promotion_overdue` | `lastPromotion / max(tenure, 1)` | share of career without advancement |

The result is then IQR-capped again — ratios explode on edge inputs.

In [ ]:
engineered = engineer_features(cleaned)
engineered[DERIVED_FEATURES].describe().T

In [ ]:
assert np.isfinite(engineered[DERIVED_FEATURES]).all().all(), 'non-finite value survived'
print('All derived features finite.')

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, col in zip(axes.ravel(), DERIVED_FEATURES):
    for label, name, colour in [(0, 'Stayed', '#52c41a'), (1, 'Left', '#ff4d4f')]:
        sns.kdeplot(engineered.loc[engineered['attrition'] == label, col], ax=ax,
                    label=name, fill=True, alpha=0.35, color=colour)
    ax.set_title(col); ax.set_xlabel(''); ax.legend()
plt.tight_layout()
plt.show()

### Do the engineered features add anything?

Only worth keeping if they separate the classes at least as well as the raw features they
are built from.

In [ ]:
rows = []
for col in ALL_FEATURES:
    a = engineered.loc[engineered['attrition'] == 1, col]
    b = engineered.loc[engineered['attrition'] == 0, col]
    pooled = np.sqrt((a.var() + b.var()) / 2)
    rows.append({'feature': col,
                 'kind': 'derived' if col in DERIVED_FEATURES else 'base',
                 'cohens_d': (a.mean() - b.mean()) / pooled if pooled else 0.0})

sig = pd.DataFrame(rows)
sig = sig.reindex(sig['cohens_d'].abs().sort_values(ascending=False).index)
sig

In [ ]:
o = sig.iloc[::-1]
plt.figure(figsize=(9, 6))
plt.barh(o['feature'], o['cohens_d'],
         color=['#722ed1' if k == 'derived' else '#1677ff' for k in o['kind']])
plt.axvline(0, color='#555', lw=1)
plt.xlabel("Cohen's d")
plt.title('Separation: base (blue) vs engineered (purple)')
plt.tight_layout()
plt.show()

### Scaling

`StandardScaler` fitted here and persisted to `artifacts/scaler.joblib`. Prediction reuses
the *fitted* scaler — refitting at inference time would be a leakage bug and would shift
every score.

In [ ]:
scaled, scaler = scale_features(engineered)

summary = pd.DataFrame({'mean': scaled[ALL_FEATURES].mean(),
                        'std': scaled[ALL_FEATURES].std()})
print(summary.round(6))
assert summary['mean'].abs().max() < 1e-6, 'features not centred'
print('\nAll features centred at 0 with unit variance.')
print('attrition preserved as 0/1:', sorted(scaled["attrition"].unique()))

## Stage 4 — Validate

In [ ]:
report = validate(scaled)
print(json.dumps(report, indent=2, default=str) if isinstance(report, dict) else report)

## Feature order is load-bearing

The model indexes features positionally. If `ALL_FEATURES` order ever changes without
retraining, predictions become silently wrong — pinned by
`TestScaleFeatures::test_feature_order_is_stable`.

In [ ]:
import joblib
saved = joblib.load(root / 'app' / 'artifacts' / 'feature_names.joblib')
print(saved)
assert list(saved) == ALL_FEATURES == list(scaled.columns[:len(ALL_FEATURES)])
print('\nOrder matches across code, artifact, and output frame.')

---
## Takeaways

1. Cleaning is **corrective, not destructive** — values are clipped into valid ranges and
   only exact duplicates and label-less rows are dropped.
2. Every divisor in feature engineering is clipped, so no `inf` reaches the scaler.
3. The scaler is **fitted once and persisted**; inference transforms with it rather than
   refitting.
4. Feature order is asserted in three places because a silent reorder is unrecoverable.

Next: `03_model_experiments.ipynb`.